In [ ]:
# Cell 1: Verify GPU Hardware
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device:", torch.cuda.get_device_name(0))
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ Please enable GPU in Colab: Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# Cell 2: Clone Repo & Apply Sinha-expt Cache Optimization Patch
%cd /content
!rm -rf GPU-talored-ANN
!git clone https://github.com/loulankxh/GPU-talored-ANN.git

# 1. Install dependencies & RAFT C++ headers
!apt-get update -qq
!apt-get install -y libboost-program-options-dev libfmt-dev nlohmann-json3-dev
!if [ ! -d "/content/raft" ]; then git clone --depth 1 -b branch-24.04 https://github.com/rapidsai/raft.git /content/raft; fi

import os
os.environ["PATH"] = "/usr/local/cuda/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

# 2. Create bucket_reordered.cu from bucket.cu with the Sinha-expt Cache Optimization
%cd /content/GPU-talored-ANN/bucketDemo/buildBucket/src
!cp bucket.cu bucket_reordered.cu

patch_code = """--- a/bucketDemo/buildBucket/src/bucket.cu
+++ b/bucketDemo/buildBucket/src/bucket_reordered.cu
@@ -2144,6 +2144,23 @@ build_vector_knn_with_tensorcore(
                          std::numeric_limits<float>::infinity());
     }
 
+    // ================================================================
+    // [Cache-Opt] Step 0.5: Compute Contiguous Bucket Offsets & Permutations
+    // ================================================================
+    std::vector<int64_t> bucket_offsets(n_centroids + 1, 0);
+    std::vector<int32_t> perm_order(N);
+    std::vector<int32_t> inverse_perm(N);
+    int64_t cur_ofs = 0;
+    for (int64_t c = 0; c < n_centroids; ++c) {
+        bucket_offsets[c] = cur_ofs;
+        for (int32_t gid : buckets[c]) {
+            perm_order[cur_ofs] = gid;
+            inverse_perm[gid] = static_cast<int32_t>(cur_ofs);
+            cur_ofs++;
+        }
+    }
+    bucket_offsets[n_centroids] = cur_ofs;
+
     // ================================================================
     // Step 1: cuBLAS handle 初始化
     // ================================================================
@@ -2315,11 +2332,18 @@ build_vector_knn_with_tensorcore(
         CUDA_CHECK(cudaStreamCreate(&streams[s]));
     }
 
-    // ---- 一次性上传 X_full + (uint8 时) shift → int8 + 计算所有点的 ‖x‖² (用 stream 0) ----
-    // norms 必须从 GEMM 实际看到的数据视图算出来（uint8 路径下要在 shift 之后用 int8 视图算）。
+    // ---- 一次性上传 X_reordered + (uint8 时) shift → int8 + 计算所有点的 ‖x‖² (用 stream 0) ----
+    // [Cache-Opt] 数据集按 bucket_offsets 顺序重排，使得同一 bucket 及近邻 bucket 在内存中连续
     {
         auto t0 = std::chrono::high_resolution_clock::now();
-        CUDA_CHECK(cudaMemcpyAsync(d_X_full, X_full, bytes_X,
+        std::vector<DataT> X_reordered(static_cast<size_t>(N) * D);
+        #pragma omp parallel for schedule(static)
+        for (int64_t i = 0; i < N; ++i) {
+            int32_t old_id = perm_order[i];
+            std::memcpy(&X_reordered[i * D], &X_full[static_cast<int64_t>(old_id) * D], D * sizeof(DataT));
+        }
+
+        CUDA_CHECK(cudaMemcpyAsync(d_X_full, X_reordered.data(), bytes_X,
                                    cudaMemcpyHostToDevice, streams[0]));
         int threads = 256;
 
@@ -2334,7 +2358,6 @@ build_vector_knn_with_tensorcore(
 
         int64_t blocks = (N + threads - 1) / threads;
         if constexpr (kIsInt8Path) {
-            // 用 int8 视图算 norms（uint8 已 shift；int8 直接）
             compute_row_norms_kernel<int8_t><<<blocks, threads, 0, streams[0]>>>( 
                 reinterpret_cast<const int8_t*>(d_X_full), d_norms_full, N, D);
         } else {
@@ -2345,7 +2368,7 @@ build_vector_knn_with_tensorcore(
         CUDA_CHECK(cudaStreamSynchronize(streams[0]));
         auto t1 = std::chrono::high_resolution_clock::now();
         double upload_ms = std::chrono::duration<double, std::milli>(t1 - t0).count();
-        std::cout << "  [VectorKNN] X_full upload + norms: " << upload_ms << " ms\n";
+        std::cout << "  [VectorKNN] [Cache-Opt] X_reordered upload + norms: " << upload_ms << " ms\n";
     }
 
     // ================================================================
@@ -2444,37 +2467,47 @@ build_vector_knn_with_tensorcore(
                                    pool_size * sizeof(int32_t),
                                    cudaMemcpyHostToDevice, streams[slot]));
 
-        // ---- 3d: GPU gather A / B / norms_pool ----
-        // INT8 路径: gather_rows_raw 不转换 type；FP32 路径: gather_rows_int32 cast 到 fp32
+        // ---- 3d: [Cache-Opt] Contiguous Slice Copies for A / B / norms_pool ----
+        // 避免 scattered gather kernel，直接按 bucket_offsets 执行连续内存切片拷贝
         {
-            int threads = 256;
-            int64_t total_A  = static_cast<int64_t>(bucket_size) * D;
-            int64_t blocks_A = (total_A + threads - 1) / threads;
-            int64_t total_B  = static_cast<int64_t>(pool_size) * D;
-            int64_t blocks_B = (total_B + threads - 1) / threads;
-
-            if constexpr (kIsInt8Path) {
-                // d_X_full 已含 int8 byte pattern (uint8 path 已在 shift 阶段就位)
-                auto d_X_int8 = reinterpret_cast<const int8_t*>(d_X_full);
-                gather_rows_raw<int8_t><<<blocks_A, threads, 0, streams[slot]>>>( 
-                    d_X_int8, d_ids_bucket[slot],
-                    reinterpret_cast<int8_t*>(d_A[slot]), bucket_size, D);
-                gather_rows_raw<int8_t><<<blocks_B, threads, 0, streams[slot]>>>( 
-                    d_X_int8, d_ids_pool[slot],
-                    reinterpret_cast<int8_t*>(d_B[slot]), pool_size, D);
-            } else {
-                gather_rows_int32<DataT><<<blocks_A, threads, 0, streams[slot]>>>( 
-                    d_X_full, d_ids_bucket[slot],
-                    reinterpret_cast<float*>(d_A[slot]), bucket_size, D);
-                gather_rows_int32<DataT><<<blocks_B, threads, 0, streams[slot]>>>( 
-                    d_X_full, d_ids_pool[slot],
-                    reinterpret_cast<float*>(d_B[slot]), pool_size, D);
-            }
+            size_t cur_pool_offset = 0;
+            auto copy_bucket_slice = [&](int64_t b_idx) {
+                int64_t b_start = bucket_offsets[b_idx];
+                int64_t b_len = bucket_offsets[b_idx + 1] - b_start;
+                if (b_len <= 0) return;
+
+                size_t byte_count = static_cast<size_t>(b_len) * D * sizeof(GemmInT);
+                CUDA_CHECK(cudaMemcpyAsync(
+                    reinterpret_cast<uint8_t*>(d_B[slot]) + cur_pool_offset * D * sizeof(GemmInT),
+                    reinterpret_cast<const uint8_t*>(d_X_full) + b_start * D * sizeof(GemmInT),
+                    byte_count,
+                    cudaMemcpyDeviceToDevice, streams[slot]));
+
+                CUDA_CHECK(cudaMemcpyAsync(
+                    d_norms_pool[slot] + cur_pool_offset,
+                    d_norms_full + b_start,
+                    b_len * sizeof(float),
+                    cudaMemcpyDeviceToDevice, streams[slot]));
+
+                cur_pool_offset += b_len;
+            };
+
+            // 1. A 矩阵直接连续拷贝当前 bucket
+            int64_t c_start = bucket_offsets[c];
+            CUDA_CHECK(cudaMemcpyAsync(
+                d_A[slot],
+                reinterpret_cast<const uint8_t*>(d_X_full) + c_start * D * sizeof(GemmInT),
+                static_cast<size_t>(bucket_size) * D * sizeof(GemmInT),
+                cudaMemcpyDeviceToDevice, streams[slot]));
+
+            // 2. B 矩阵按邻居 bucket 拼接连续切片
+            copy_bucket_slice(c);
+            for (uint32_t k = 0; k < K; ++k) {
+                uint32_t nb_c = centroid_knn_graph[c * K + k];
+                if (nb_c < static_cast<uint32_t>(n_centroids) && nb_c != static_cast<uint32_t>(c)) {
+                    copy_bucket_slice(nb_c);
+                }
+            }
         }
"""
with open("/tmp/cache_opt.patch", "w") as f:
    f.write(patch_code)

!patch /content/GPU-talored-ANN/bucketDemo/buildBucket/src/bucket_reordered.cu < /tmp/cache_opt.patch

# 3. Update CMakeLists.txt
cmake_patch = """
if(NOT TARGET raft::raft)
    add_library(raft_interface INTERFACE)
    add_library(raft::raft ALIAS raft_interface)
endif()
include_directories(SYSTEM /content/raft/cpp/include /content/raft/cpp/build/_deps/nlohmann_json-src/include)

add_executable(bucket_reordered ${BUILD_SRC_PATH}/bucket_reordered.cu)
target_link_libraries(bucket_reordered PRIVATE CUDA::cudart CUDA::cublas OpenMP::OpenMP_CXX raft::raft fmt boost_program_options)
"""
with open("/content/GPU-talored-ANN/bucketDemo/buildBucket/CMakeLists.txt", "a") as f:
    f.write(cmake_patch)

print("✅ Sinha-expt cache optimization successfully applied!")

In [ ]:
# Cell 3: Configure & Build Both Binaries (bucket2 & bucket_reordered)
!rm -rf /content/GPU-talored-ANN/bucketDemo/buildBucket/build
!mkdir -p /content/GPU-talored-ANN/bucketDemo/buildBucket/build
%cd /content/GPU-talored-ANN/bucketDemo/buildBucket/build

!cmake .. \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CXX_COMPILER=/usr/bin/g++ \
    -DCMAKE_CUDA_COMPILER=/usr/local/cuda/bin/nvcc \
    -DCMAKE_CUDA_ARCHITECTURES=75

!make -j$(nproc) bucket2 bucket_reordered

In [ ]:
# Cell 4: Run the Benchmark Suite
%cd /content/GPU-talored-ANN/bucketDemo

bench_code = """import os, re, subprocess, argparse, struct, numpy as np, matplotlib.pyplot as plt
def gen_data(p, N, D):
    if os.path.exists(p) and os.path.getsize(p) == (8 + N * D * 4): return
    os.makedirs(os.path.dirname(p) or ".", exist_ok=True)
    np.random.seed(42)
    with open(p, "wb") as f:
        f.write(struct.pack("ii", N, D))
        f.write(np.random.randn(N, D).astype(np.float32).tobytes())

def run(bin_p, d_p, out_p, k, m):
    os.makedirs(out_p, exist_ok=True)
    cmd = [bin_p, "-i", d_p, "-o", out_p, "--knn-k", str(k), "--neighbors-m", str(m), "-t", "1"]
    p = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    out = p.stdout
    g = re.search(r"pure GEMM:\\s*([\\d\\.]+)\\s*ms", out)
    t = re.search(r"Total pipeline elapsed:\\s*([\\d\\.]+)\\s*s", out)
    return float(t.group(1)) if t else 0.0, float(g.group(1)) if g else 0.0

sizes = [50000, 100000, 200000]
b_t, o_t, b_g, o_g = [], [], [], []
for N in sizes:
    dp = f"test_data/vectors_{N//1000}k_128d.fbin"
    gen_data(dp, N, 128)
    bt, bg = run("./buildBucket/build/bucket2", dp, f"output/base_{N}", 32, 32)
    ot, og = run("./buildBucket/build/bucket_reordered", dp, f"output/opt_{N}", 32, 32)
    b_t.append(bt); o_t.append(ot); b_g.append(bg); o_g.append(og)
    sp = bt / max(1e-6, ot)
    print(f"N={N:,} | Baseline: {bt:.2f}s | Reordered: {ot:.2f}s | Speedup: {sp:.2f}x")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4), dpi=150)
ax1.plot([s//1000 for s in sizes], b_t, "o-", label="Baseline (bucket2)")
ax1.plot([s//1000 for s in sizes], o_t, "s-", label="Cache-Opt (bucket_reordered)")
ax1.set_title("Total Indexing Time (s)"); ax1.set_xlabel("Size (K)"); ax1.legend(); ax1.grid(True)
ax2.bar([f"{s//1000}K" for s in sizes], [b/max(1e-6, o) for b, o in zip(b_t, o_t)], color="#5cb85c")
ax2.axhline(1.0, color="gray", linestyle=":"); ax2.set_title("Speedup Factor (x)"); ax2.grid(True)
plt.tight_layout()
os.makedirs("output/benchmark", exist_ok=True)
plt.savefig("output/benchmark/benchmark_results.png")
print("Saved graph to output/benchmark/benchmark_results.png")
"""
with open("benchmark.py", "w") as f:
    f.write(bench_code)

!python3 benchmark.py

In [ ]:
# Cell 5: Display Benchmark Results Graph
from IPython.display import Image, display
display(Image("/content/GPU-talored-ANN/bucketDemo/output/benchmark/benchmark_results.png"))